# Import libraries

In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector
from sklearn.metrics import pairwise_distances
from scipy.stats import pearsonr, spearmanr

import numpy as np 

# Functions

In [ ]:
def get_high_dim_data_ML(table_name, EM_name, ML_val, additional_info): 

    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    register_vector(conn); 
    cur = conn.cursor()
    # cur.execute(f"SELECT embedding_data FROM {table_name} WHERE embedding_max_length = {ML_val}")  

    cur.execute(f"""
        SELECT embedding_data
        FROM {table_name}
        WHERE embedding_model = %s AND embedding_max_length = %s AND additional_info_inc = %s
    """, (EM_name, ML_val, additional_info))  


    embedding_arr = np.array([row[0] for row in cur.fetchall()])
    return embedding_arr

In [ ]:
def get_low_dim_data(EM_name, ML_val, DRM, dim_size, additional_info): 

    embedding_arr               = []
    sv_dataset_size             = 16917
    dr_method                   = "PCA"

    # establish a connection to the database
    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    cur = conn.cursor()

    # get the low-dim embedding data
    cur.execute("""
        SELECT low_dim_embedding
        FROM low_dim_embeddings_20ng
        WHERE embedding_model = %s AND embedding_max_length = %s AND dr_method = %s AND dimension_size = %s AND additional_info = %s
    """, (EM_name, ML_val, DRM, dim_size, additional_info))  

    result = cur.fetchone()

    for i in range(sv_dataset_size): 
        embedding = result[0][i]
        embedding_arr.append(embedding)

    cur.close()
    conn.close()

    return embedding_arr

In [ ]:
def get_low_dim_data_UMAP(EM_name, ML_val, DRM, dim_size, additional_info, dr_parameter): 

    embedding_arr               = []
    sv_dataset_size             = 16917
    dr_method                   = "UMAP"

    # establish a connection to the database
    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    cur = conn.cursor()

    # get the low-dim embedding data
    cur.execute("""
        SELECT low_dim_embedding
        FROM low_dim_embeddings_20ng
        WHERE embedding_model = %s AND embedding_max_length = %s AND dr_method = %s AND dimension_size = %s AND additional_info = %s AND dr_parameter = %s
    """, (EM_name, ML_val, DRM, dim_size, additional_info, dr_parameter))  

    result = cur.fetchone()

    for i in range(sv_dataset_size): 
        embedding = result[0][i]
        embedding_arr.append(embedding)

    cur.close()
    conn.close()

    return embedding_arr

In [ ]:
def get_low_dim_data_PaCMAP(EM_name, ML_val, DRM, dim_size, additional_info, dr_parameter, mn_ratio, fp_ratio): 

    embedding_arr               = []
    sv_dataset_size             = 16917
    dr_method                   = "PaCMAP"

    # establish a connection to the database
    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    cur = conn.cursor()

    # get the low-dim embedding data
    cur.execute("""
        SELECT low_dim_embedding
        FROM low_dim_embeddings_20ng
        WHERE embedding_model = %s AND embedding_max_length = %s AND dr_method = %s AND dimension_size = %s AND additional_info = %s AND dr_parameter = %s AND mn_ratio = %s AND fp_ratio = %s
    """, (EM_name, ML_val, DRM, dim_size, additional_info, dr_parameter, mn_ratio, fp_ratio))  

    result = cur.fetchone()

    for i in range(sv_dataset_size): 
        embedding = result[0][i]
        embedding_arr.append(embedding)

    cur.close()
    conn.close()

    return embedding_arr

In [ ]:
def organise_pairwise_distance_information(pairwise_distance_data): 

    # checks if the pairwise distance data is in a list or that it's shape is 1, 
    # to see if its a vector arr
    if isinstance(pairwise_distance_data, list) or len(pairwise_distance_data.shape) == 1:

        # transform the data into a square form format
        pairwise_data = spatial.distance.squareform(pairwise_distance_data)
    else:

        # otherwise it is already in that format, and return it as is
        pairwise_data = pairwise_distance_data
    
    return pairwise_data

# function to calculate the nearest number of neigbours for each datapoints
def calc_nearest_neighbours(pairwise_data, number_neighbours):

    # organise the pairwise distance array to find the number of nearest neighbours for each datapoint
    nn_data = pairwise_data.argsort()

    # extract the number of nearest neighbours close to this point
    knn_data = nn_data[:, :number_neighbours + 1][:, 1:]

    # return the neighbouring datapoints
    return nn_data, knn_data

In [ ]:
# metrics taken from paper code below:
# Daniel Atzberger, Tim Cech, Willy Scheibel, Jürgen Döllner, Michael Behrisch, and Tobias Schreck. 2025b. A large-scale sensitivity analysis on latent 
# embeddings and dimensionality reductions for text spatializations. IEEE Transactions on Visualization and Computer Graphics, 31(1):305–315

# functions to calculate local metrics

# calculate trustworthiness metric
def calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, number_neighbours): 

    # tally value
    sum_i = 0

    # k is the number of neighbours, n is the number of datapoints
    n = num_datapoints
    k = number_neighbours

    # for every datapoint 
    for i in range(n):

        # find the neighbours which are in the low_dim datapoint, but not in the 
        # high_dim data - calculate false neighbours
        U = np.setdiff1d(low_dim_knn_data[i], high_dim_knn_data[i])

        # calculate the penalty for the false neighbours
        sum_j = 0
        for j in range(U.shape[0]):
            sum_j += np.where(high_dim_nn_data[i] == U[j])[0] - k

        # add this penalty
        sum_i += sum_j
        
    try:
        trustworthiness = float((1 - (2 / (n * k * (2 * n - 3 * k - 1)) * sum_i)).squeeze())
    except AttributeError:  # Everything stayed constant
        trustworthiness = 1.0

    return trustworthiness

# calculate continuity metric
def calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, number_neighbours): 

    # tally value
    sum_i = 0

    # k is the number of neighbours, n is the number of datapoints
    n = num_datapoints
    k = number_neighbours

    # for every datapoint 
    for i in range(n):
        V = np.setdiff1d(high_dim_knn_data[i], low_dim_knn_data[i])

        sum_j = 0
        for j in range(V.shape[0]):
            sum_j += np.where(low_dim_nn_data[i] == V[j])[0] - k

        sum_i += sum_j

    try:
        continuity = float((1 - (2 / (n * k * (2 * n - 3 * k - 1)) * sum_i)).squeeze())
    except AttributeError:  # Everything stayed the same
        continuity = 1.0

    return continuity

In [ ]:
def add_metric_info_20NG(embedding_model_name, additional_info, embedding_model_max_length, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dimension_size_val, trust_ten_neigh_val, trust_fifty_neigh_val, trust_hundred_neigh_val, cont_ten_neigh_val, cont_fifty_neigh_val, cont_hundred_neigh_val, pearson_val, spearman_val): 

    # include variables
    dataset_info            = '20NG'

    # Connect to PostgreSQL
    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    cursor = conn.cursor()

    # SV PaCMAP Data
    cursor.execute(
        "INSERT INTO METRICS_TWENTY_NG (dataset,        additional_info, embedding_model,        embedding_max_length,       dr_method,      dr_parameter,       mn_ratio,       fp_ratio,       dimension_size,     trustworthiness_ten_neigh,  trustworthiness_fifty_neigh,    trustworthiness_hundred_neigh,  continuity_ten_neigh,   continuity_fifty_neigh,     continuity_hundred_neigh,   pearson_corr, spearman_corr) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)",
                                        (dataset_info,  additional_info, embedding_model_name,   embedding_model_max_length, dr_method_info, dr_parameter_val,   mn_ratio_val,   fp_ratio_val,   dimension_size_val, trust_ten_neigh_val,        trust_fifty_neigh_val,          trust_hundred_neigh_val,        cont_ten_neigh_val,     cont_fifty_neigh_val,       cont_hundred_neigh_val,     pearson_val, spearman_val)
    )

    conn.commit()
    cursor.close()
    conn.close()

In [ ]:
additional_info         = "from_organization_newsgroup_body"
dim_arr                 = [5,10,50]
local_num_neigh         = [10, 50, 100]
num_datapoints          = 16917

# PCA

In [ ]:
dr_method_info          = 'PCA'
dr_parameter_val        = 'N/A'
mn_ratio_val            = 'N/A'
fp_ratio_val            = 'N/A'

### BERT-128 

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 128
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

# print(high_dim_data[0])
high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BERT-256

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 256
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BERT-512

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 512
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[0])

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM 128

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 128
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM 256

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 256
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM 512

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 512
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5 128

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 128
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5 256

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 256
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5 512

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 512
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer 1024

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 1024
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer 2048

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 2048
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer 4096

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 4096
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird 1024

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 1024
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird 2048

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 2048
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird 4096

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 4096
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3 1024

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 1024
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3 2048

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 2048
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3 4096

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 4096
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

# UMAP

In [ ]:
dr_method_info          = 'UMAP'
dr_parameter_val        = '30'
mn_ratio_val            = 'N/A'
fp_ratio_val            = 'N/A'

### BERT-128

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 128
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BERT-256

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 256
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

# print(high_dim_data[0])
high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BERT-512

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 512
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

# print(high_dim_data[0])
high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM-128

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 128
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM-256

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 256
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM-512

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 512
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5-128

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 128
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5-256

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 256
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5-512

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 512
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer-1024

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 1024
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer-2048

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 2048
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer-4096

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 4096
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird-1024

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 1024
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird-2048

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 2048
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird-4096

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 4096
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3-1024

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 1024
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3-2048

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 2048
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3-4096

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 4096
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_UMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

# PaCMAP

In [ ]:
dr_method_info          = 'PaCMAP'
dr_parameter_val        = '30'
mn_ratio_val            = '2'
fp_ratio_val            = '0.5'

### BERT-128

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 128
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BERT-256

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 256
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

# print(high_dim_data[0])
high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BERT-512

In [ ]:
# get the high-dim data
EM_arr_label            = "google-bert/bert-base-uncased"
L_arr_label             = 512
EM_short_name           = "bert"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

# print(high_dim_data[0])
high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM-128

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 128
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM-256

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 256
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### All-MiniLM-512

In [ ]:
# get the high-dim data
EM_arr_label            = "sentence-transformers/all-MiniLM-L6-v2"
L_arr_label             = 512
EM_short_name           = "allminilm"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5-128

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 128
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5-256

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 256
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### E5-512

In [ ]:
# get the high-dim data
EM_arr_label            = "intfloat/e5-small-v2"
L_arr_label             = 512
EM_short_name           = "e5"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer-1024

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 1024
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer-2048

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 2048
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Longformer-4096

In [ ]:
# get the high-dim data
EM_arr_label            = "allenai/longformer-base-4096"
L_arr_label             = 4096
EM_short_name           = "longformer"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird-1024

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 1024
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird-2048

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 2048
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### BigBird-4096

In [ ]:
# get the high-dim data
EM_arr_label            = "google/bigbird-roberta-base"
L_arr_label             = 4096
EM_short_name           = "bigbird"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3-1024

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 1024
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3-2048

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 2048
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred

### Qwen3-4096

In [ ]:
# get the high-dim data
EM_arr_label            = "qwen3-embedding:8b"
L_arr_label             = 4096
EM_short_name           = "qwen3"
table_name              = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
high_dim_data           = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)

high_dim_pairwise_data  = pairwise_distances(high_dim_data) 
high_dim_pairwise_org   = organise_pairwise_distance_information(high_dim_pairwise_data)
high_dim_pairwise_org   = high_dim_pairwise_org.astype(np.float32)
indices                 = np.triu_indices_from(high_dim_pairwise_org, k=1)
high_dim_distances      = high_dim_pairwise_org[indices]
print("High-dim done")

In [ ]:
for i in range(len(dim_arr)): 

    low_dim_data                = get_low_dim_data_PaCMAP(EM_arr_label, L_arr_label, dr_method_info, dim_arr[i], additional_info, dr_parameter_val, mn_ratio_val, fp_ratio_val)
    low_dim_pairwise            = pairwise_distances(low_dim_data)
    low_dim_pairwise_org        = organise_pairwise_distance_information(low_dim_pairwise)
    low_dim_pairwise_org        = low_dim_pairwise_org.astype(np.float32)
    low_dim_distances           = low_dim_pairwise_org[indices]
    print("Low-dim done")
        
    # global metrics
    pearson_val, pearson_pval   = pearsonr(high_dim_distances, low_dim_distances)
    print("Pearson done")

    spearman_val, spearman_pval = spearmanr(high_dim_distances,  low_dim_distances)
    print("Spearman done")

    # local metrics
    # 10 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[0])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[0])
    trustworthiness_10                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    continuity_10                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[0])
    print("T & C10")

    # 50 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[1])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[1])
    trustworthiness_50                  = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    continuity_50                       = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[1])
    print("T & C50")

    # 100 neigh
    high_dim_nn_data, high_dim_knn_data = calc_nearest_neighbours(high_dim_pairwise_org, local_num_neigh[2])
    low_dim_nn_data, low_dim_knn_data   = calc_nearest_neighbours(low_dim_pairwise_org, local_num_neigh[2])
    trustworthiness_100                 = calculate_trustworthiness(high_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    continuity_100                      = calculate_continuity(low_dim_nn_data, high_dim_knn_data, low_dim_knn_data, num_datapoints, local_num_neigh[2])
    print("T & C100")

    # round the calculate metrics to 4 d.p. 
    pearson_round                       = round(pearson_val, 4)
    spearman_round                      = round(spearman_val, 4)
    trustworthiness_ten                 = round(trustworthiness_10, 4)
    trustworthiness_fifty               = round(trustworthiness_50, 4)
    trustworthiness_hundred             = round(trustworthiness_100, 4)
    continuity_ten                      = round(continuity_10, 4)
    continuity_fifty                    = round(continuity_50, 4)
    continuity_hundred                  = round(continuity_100, 4)
    print("rounded")

    # add to the database
    add_metric_info_20NG(EM_arr_label, additional_info, L_arr_label, dr_method_info, dr_parameter_val, mn_ratio_val, fp_ratio_val, dim_arr[i], float(trustworthiness_ten), float(trustworthiness_fifty), float(trustworthiness_hundred), float(continuity_ten), float(continuity_fifty), float(continuity_hundred), float(pearson_round), float(spearman_round))
    print("added")

    del low_dim_data
    del low_dim_pairwise
    del low_dim_pairwise_org
    del low_dim_distances
    del pearson_val
    del pearson_pval
    del spearman_val
    del spearman_pval
    del trustworthiness_10
    del trustworthiness_50
    del trustworthiness_100
    del continuity_10
    del continuity_50
    del continuity_100
    del pearson_round
    del spearman_round
    del trustworthiness_ten
    del trustworthiness_fifty
    del trustworthiness_hundred
    del continuity_ten
    del continuity_fifty
    del continuity_hundred